## Final Driver Table 2024

In [84]:
import pandas as pd
import numpy as np
import altair as alt

pd.set_option("display.max_columns", None)

drivers_df = pd.read_csv("../../data/ergast/drivers.csv")
lap_times_df = pd.read_csv("../../data/ergast/lap_times.csv")
pit_stops_df = pd.read_csv("../../data/ergast/pit_stops.csv")
races_df = pd.read_csv("../../data/ergast/races.csv")
results_df = pd.read_csv("../../data/ergast/results.csv")
constructors_df = pd.read_csv("../../data/ergast/constructors.csv")

driver_columns_to_keep = ["driverId", "forename", "surname", "nationality"]
drivers_df = drivers_df[driver_columns_to_keep]

# Keep only races 2009 and on
races_df = races_df[races_df["year"] == 2024]
races_df = races_df[["raceId", "year"]]

results_df = results_df[["raceId" ,"driverId" ,"constructorId", "position", "points"]]

constructor_df = constructors_df[["constructorId", "name"]]
constructor_df = constructor_df.rename(columns = {"name": "constructor_name"})



final_table_df = pd.merge(races_df, results_df, how = "left", on = "raceId")
final_table_df = pd.merge(final_table_df, drivers_df, how = "left", on = "driverId")
final_table_df = pd.merge(final_table_df, constructor_df, how = "left", on = "constructorId")


# Group by driverId and compute the total points for each driver
driver_table_df = final_table_df.groupby(["driverId", "forename", "surname", "constructorId", "constructor_name"], as_index = False)["points"].sum()

driver_table_df["rank"] = driver_table_df["points"].rank(method = "min", ascending = False)
driver_table_df = driver_table_df.rename(columns = {"points": "total_points"})

# Optionally sort the results by total points in descending order
driver_table_df = driver_table_df.sort_values(by = "total_points", ascending = False)

# Specify the desired column order
desired_order = ["rank", "forename", "surname", "constructor_name", "total_points"]

# Reorder the DataFrame columns
driver_table_df = driver_table_df[desired_order]

driver_table_df = driver_table_df[driver_table_df["rank"] <= 10].reset_index(drop = True)

driver_table_df["rank"] = driver_table_df["rank"].astype(int)

In [85]:
driver_table_df.head(10)

,rank,forename,surname,constructor_name,total_points
0,1,Max,Verstappen,Red Bull,399.0
1,2,Lando,Norris,McLaren,344.0
2,3,Charles,Leclerc,Ferrari,327.0
3,4,Oscar,Piastri,McLaren,265.0
4,5,Carlos,Sainz,Ferrari,262.0
5,6,George,Russell,Mercedes,226.0
6,7,Lewis,Hamilton,Mercedes,207.0
7,8,Sergio,Pérez,Red Bull,138.0
8,9,Fernando,Alonso,Aston Martin,70.0
9,10,Pierre,Gasly,Alpine F1 Team,40.0


In [86]:
#driver_table_df.to_csv("../../data/2024_final_driver_table.csv")

In [87]:
# Optional: create full name column
driver_table_df["driver_name"] = driver_table_df["forename"] + " " + driver_table_df["surname"]

# Drop separate forename/surname if you want a cleaner table
driver_table_df = driver_table_df[["rank", "driver_name", "constructor_name", "total_points"]]

# Rename columns for better grammar
driver_table_df = driver_table_df.rename(columns = {
    "rank": "Rank",
    "driver_name": "Driver Name",
    "constructor_name": "Constructor",
    "total_points": "Total Points"
})

# Function to highlight Gold, Silver, and Bronze for Total Points
def highlight_row(row):
    if row["Rank"] == driver_table_df["Rank"].min():
        return ['background-color: #FFB800; font-weight: bold; color: white'] * len(row)  # Darker Gold
    elif row["Rank"] == driver_table_df["Rank"].nsmallest(2).iloc[-1]:
        return ['background-color: #A9A9A9; font-weight: bold; color: white'] * len(row)  # Darker Silver
    elif row["Rank"] == driver_table_df["Rank"].nsmallest(3).iloc[-1]:
        return ['background-color: #C87F4D; font-weight: bold; color: white'] * len(row)  # Darker Bronze
    else:
        return [''] * len(row)  # No styling for others

# Style the table for the top 10 drivers with row highlights
styled_table = driver_table_df.style \
    .apply(highlight_row, axis = 1) \
    .format({"Total Points": "{:.0f}"}) \
    .set_properties(**{
        "text-align": "left",
        "font-size": "18px",
        "color": "white",
        "padding": "5px",
        "font-family": "'Quicksand', sans-serif"
    }) \
    .set_table_styles([{
            "selector": "th, td",
            "props": [("width", "130px")]
        }, {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-size", "20px"),
                ("color", "white"),
                ("padding", "5px"),
                ("border-bottom", "2px solid white"),  # Solid bottom under header
                ("border-top", "none"),
                ("border-left", "none"),
                ("border-right", "none"),
                ("font-family", "'Quicksand', sans-serif")
            ]
        }, {
            "selector": "td",
            "props": [
                ("text-align", "left"),
                ("font-size", "18px"),
                ("color", "white"),
                ("padding", "5px"),
                ("border-bottom", "1px dashed white"),  # Dashed between rows
                ("border-top", "none"),
                ("border-left", "none"),
                ("border-right", "none"),
                ("font-family", "'Quicksand', sans-serif")
            ]
        }, {
            "selector": "td:nth-child(1)",
            "props": [
                ("text-align", "center")
            ]
        }, {
            "selector": "td:nth-child(2)",
            "props": [
                ("white-space", "nowrap"),
                ("overflow", "hidden"),
                ("text-overflow", "ellipsis")
            ]
        }, {
            "selector": "td:nth-child(4)",
            "props": [
                ("white-space", "nowrap"),
                ("overflow", "hidden"),
                ("text-overflow", "ellipsis"),
                ("text-align", "center")
            ]
        }, {
            "selector": "caption",
            "props": [
                ("font-weight", "bold"),
                ("font-size", "20px"),
                ("margin-bottom", "5px"),
                ("color", "white"),
                ("padding", "5px"),
                ("font-family", "'Quicksand', sans-serif")
            ]
        }
    ]) \
    .set_caption("2024 F1 Driver Standings (Top 10 Drivers)") \
    .hide(axis = "index")

styled_table




Rank,Driver Name,Constructor,Total Points
1,Max Verstappen,Red Bull,399
2,Lando Norris,McLaren,344
3,Charles Leclerc,Ferrari,327
4,Oscar Piastri,McLaren,265
5,Carlos Sainz,Ferrari,262
6,George Russell,Mercedes,226
7,Lewis Hamilton,Mercedes,207
8,Sergio Pérez,Red Bull,138
9,Fernando Alonso,Aston Martin,70
10,Pierre Gasly,Alpine F1 Team,40


In [79]:
#styled_table.to_html("../../img/2024_driver_standings_table.html")

## Final Constructor Table 2024

In [80]:
import pandas as pd
import numpy as np
import altair as alt

pd.set_option("display.max_columns", None)

drivers_df = pd.read_csv("../../data/ergast/drivers.csv")
lap_times_df = pd.read_csv("../../data/ergast/lap_times.csv")
pit_stops_df = pd.read_csv("../../data/ergast/pit_stops.csv")
races_df = pd.read_csv("../../data/ergast/races.csv")
results_df = pd.read_csv("../../data/ergast/results.csv")
constructors_df = pd.read_csv("../../data/ergast/constructors.csv")

driver_columns_to_keep = ["driverId", "forename", "surname", "nationality"]
drivers_df = drivers_df[driver_columns_to_keep]

# Keep only races 2009 and on
races_df = races_df[races_df["year"] == 2024]
races_df = races_df[["raceId", "year"]]

results_df = results_df[["raceId" ,"driverId" ,"constructorId", "position", "points"]]

constructor_df = constructors_df[["constructorId", "name"]]
constructor_df = constructor_df.rename(columns = {"name": "constructor_name"})



final_table_df = pd.merge(races_df, results_df, how = "left", on = "raceId")
final_table_df = pd.merge(final_table_df, drivers_df, how = "left", on = "driverId")
final_table_df = pd.merge(final_table_df, constructor_df, how = "left", on = "constructorId")


# Group by driverId and compute the total points for each driver
constructor_table_df = final_table_df.groupby(["constructorId", "constructor_name"], as_index = False)["points"].sum()

constructor_table_df["rank"] = constructor_table_df["points"].rank(method = "min", ascending = False)
constructor_table_df = constructor_table_df.rename(columns = {"points": "total_points"})

# Optionally sort the results by total points in descending order
constructor_table_df = constructor_table_df.sort_values(by = "total_points", ascending = False)

# Specify the desired column order
desired_order = ["rank", "constructor_name", "total_points"]

# Reorder the DataFrame columns
constructor_table_df = constructor_table_df[desired_order]

constructor_table_df = constructor_table_df[constructor_table_df["rank"] <= 10].reset_index(drop = True)

constructor_table_df["rank"] = constructor_table_df["rank"].astype(int)

In [81]:
constructor_table_df.head(10)

,rank,constructor_name,total_points
0,1,McLaren,609.0
1,2,Ferrari,595.0
2,3,Red Bull,537.0
3,4,Mercedes,433.0
4,5,Aston Martin,94.0
5,6,Alpine F1 Team,63.0
6,7,Haas F1 Team,51.0
7,8,RB F1 Team,40.0
8,9,Williams,17.0
9,10,Sauber,4.0


In [82]:
#constructor_table_df.to_csv("../../data/2024_final_constructor_table.csv")

In [ ]:
# Rename columns for better grammar
constructor_table_df = constructor_table_df.rename(columns = {
    "rank": "Rank",
    "constructor_name": "Constructor",
    "total_points": "Total Points"
})

# Function to highlight entire row for Gold, Silver, and Bronze based on Rank for constructors
def highlight_rank(row):
    if row["Rank"] == constructor_table_df["Rank"].min():
        return ['background-color: #FFB800; font-weight: bold; color: white'] * len(row)  # Darker Gold
    elif row["Rank"] == constructor_table_df["Rank"].nsmallest(2).iloc[-1]:
        return ['background-color: #A9A9A9; font-weight: bold; color: white'] * len(row)  # Darker Silver
    elif row["Rank"] == constructor_table_df["Rank"].nsmallest(3).iloc[-1]:
        return ['background-color: #C87F4D; font-weight: bold; color: white'] * len(row)  # Darker Bronze
    else:
        return [''] * len(row)  # No styling for others

# Style the table for constructors with row-based highlights
styled_constructor_table = constructor_table_df.style \
    .apply(highlight_rank, axis = 1) \
    .format({"Total Points": "{:.0f}"}) \
    .set_properties(**{
        "text-align": "left",
        "font-size": "18px",
        "color": "white",
        "padding": "5px",
        "font-family": "'Quicksand', sans-serif"
    }) \
    .set_table_styles([{
            "selector": "th, td",
            "props": [("width", "145px")]
        }, {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-size", "20px"),
                ("color", "white"),
                ("padding", "5px"),
                ("border-bottom", "2px solid white"),  # Solid bottom border for headers
                ("border-top", "none"),
                ("border-left", "none"),
                ("border-right", "none"),
                ("font-family", "'Quicksand', sans-serif")
            ]
        }, {
            "selector": "td",
            "props": [
                ("text-align", "left"),
                ("font-size", "18px"),
                ("color", "white"),
                ("padding", "5px"),
                ("border-bottom", "1px dashed white"),  # Dashed between rows
                ("border-top", "none"),
                ("border-left", "none"),
                ("border-right", "none"),
                ("font-family", "'Quicksand', sans-serif")
            ]
        }, {
            "selector": "td:nth-child(1)",
            "props": [
                ("text-align", "center")
            ]
        }, {
            "selector": "td:nth-child(2)",
            "props": [
                ("white-space", "nowrap"),
                ("overflow", "hidden"),
                ("text-overflow", "ellipsis")
            ]
        }, {
            "selector": "td:nth-child(3)",
            "props": [
                ("white-space", "nowrap"),
                ("overflow", "hidden"),
                ("text-overflow", "ellipsis"),
                ("text-align", "center")
            ]
        }, {
            "selector": "caption",
            "props": [
                ("font-weight", "bold"),
                ("font-size", "20px"),
                ("margin-bottom", "5px"),
                ("color", "white"),
                ("padding", "5px"),
                ("font-family", "'Quicksand', sans-serif")
            ]
        }
    ]) \
    .set_caption("2024 F1 Constructor Championship Standings") \
    .hide(axis = "index")

styled_constructor_table

Rank,Constructor,Total Points
1,McLaren,609
2,Ferrari,595
3,Red Bull,537
4,Mercedes,433
5,Aston Martin,94
6,Alpine F1 Team,63
7,Haas F1 Team,51
8,RB F1 Team,40
9,Williams,17
10,Sauber,4


In [ ]:
#styled_constructor_table.to_html("../../img/2024_constructor_standings_table.html")